In [ ]:
import numpy as np
from numpy import bool_
import pandas as pd


from scipy import stats

from matplotlib import pyplot as plt    
import math  
#from bioinfokit import analys, visuz
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.backends.backend_pdf
import statsmodels
import os
import dash_bio
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
#from bioinfokit import analys, visuz
from statistics import stdev
import statistics
from statsmodels.stats.multitest import multipletests
import re
import random
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
from matplotlib import pyplot as plt
from matplotlib_venn import venn2, venn3


In [ ]:
#Mageck auto output 


#Compare overlapping hits

#Screenshots of bed file locations



#Timepoint stats from input list of genes
def get_timepoint_information(list_genes,prefix,title):
    df = pd.read_csv('Data/timepoints/combined_timepoint_data.txt',sep='\t')
    # Set the 'gene' column as the index
    df.set_index('Gene_Id', inplace=True)

    # Transpose the dataframe to have timepoints as columns
    df = df.T
        
    # Create a PDF file to save the histograms
    pdf_file = prefix + "/" + title + ".pdf"

    # Create a PDF pages object
    pdf_pages = PdfPages(pdf_file)
    
    
    for input_gene in list_genes:
        gene_data = df[input_gene]

        #Plot the line graph
        ax = gene_data.plot(kind='line', marker='o')
        plt.xlabel('Timepoints')
        plt.ylabel('Value')
        plt.title('Value Change Over Time - Gene: {}'.format(input_gene))
        
        # Set the y-axis limit to start from zero
        ax.set_ylim(ymin=0)

        # Add data labels to the graph
        for x, y in zip(range(len(gene_data)), gene_data.values):
            ax.annotate('{:.1f}'.format(y), xy=(x, y), xytext=(5, 5), textcoords='offset points')
            
        

        #Save/display the plot
        pdf_pages.savefig()
        plt.show()
        plt.close()
    
        
    
    pdf_pages.close()




#Target barplot for list of targets
def generate_target_barplot_summaries_target_specific(timepoint1,timepoint2,prefix,list_targets):
    comparison = timepoint1 + "_vs_" + timepoint2
    pdf_title = prefix+'/'+comparison+"/"+prefix+"_" + timepoint1 + "_vs_" + timepoint2 + "_target_freq_swarms.pdf"
    pdf = matplotlib.backends.backend_pdf.PdfPages(pdf_title)  
    
    df = pd.read_csv(prefix+'/'+comparison+"/"+prefix+"_" + timepoint1 + "_vs_" + timepoint2 + "_normalized_dataframe.txt",sep='\t')
    
    common_suffix = '_D0_ctrl_norm'

    
    for target in list_targets:
        print(target)
        a4_dims = (20, 8.27)
        fig, ax = plt.subplots(figsize=a4_dims)
        
        #If we want to include safe-targeting and non-targeting in plot
        # Define the patterns to match
        #patterns = [target, 'non-targeting','safe-targeting']
        # Create a regular expression pattern using the patterns list
        #pattern = '|'.join(patterns)
        #target_df = df[df['Target'].str.contains(pattern)][['OligoID','Target','D0'+common_suffix,'Pos'+common_suffix,'Neg'+common_suffix,'EC_total'+common_suffix]]


        # Filter the DataFrame based on the 'Target' column
        target_df = df[df['Target']==target][['OligoID','Target','D0'+common_suffix,'Pos'+common_suffix,'Neg'+common_suffix,'EC_total'+common_suffix]]

        if len(df[df['Target']==target])>0:
            
            column_mapping = {'OligoID': 'OligoID', 'Target': 'Target', 'D0'+common_suffix: 'D0', 'Pos'+common_suffix: 'Pos', 'Neg'+common_suffix: 'Neg','EC_total'+common_suffix: 'EC_total',}
            target_df = target_df.rename(columns=column_mapping)

            target_df = target_df.melt(id_vars=['OligoID', 'Target'], var_name='Timepoint', value_name='Frequency')
            ax = sns.barplot(data=target_df,x="Target", y="Frequency", hue="Timepoint",alpha=0.2,ci='sd',capsize=0.1)
            sns.swarmplot(x="Target", y="Frequency", data=target_df, hue="Timepoint",dodge=True,ax=ax)
            ax.set_title(target,fontsize=20)
            ax.set_ylabel('Normalized Frequency')
            pdf.savefig(fig)
        else:
            print('Missing ' + target)
    pdf.close()
    
    
def generate_target_barplot_summaries_target_specific_skew(timepoint1,timepoint2,prefix,list_targets):
    comparison = timepoint1 + "_vs_" + timepoint2
    pdf_title = prefix+'/'+comparison+"/"+prefix+"_" + timepoint1 + "_vs_" + timepoint2 + "_target_freq_swarms.pdf"
    pdf = matplotlib.backends.backend_pdf.PdfPages(pdf_title)  
    
    df = pd.read_csv(prefix+"/"+comparison + "/" + prefix+"_D0_skew_comparison.txt",sep='\t')

    
    for target in list_targets:
        print(target)
        a4_dims = (20, 8.27)
        fig, ax = plt.subplots(figsize=a4_dims)
        
        #If we want to include safe-targeting and non-targeting in plot
        # Define the patterns to match
        #patterns = [target, 'non-targeting','safe-targeting']
        # Create a regular expression pattern using the patterns list
        #pattern = '|'.join(patterns)
        #target_df = df[df['Target'].str.contains(pattern)][['OligoID','Target','D0'+common_suffix,'Pos'+common_suffix,'Neg'+common_suffix,'EC_total'+common_suffix]]


        # Filter the DataFrame based on the 'Target' column
        target_df = df[df['Target']==target][['OligoID','Target','D0_freq','Skew_freq']]

        if len(df[df['Target']==target])>0:
            
            target_df = target_df.melt(id_vars=['OligoID', 'Target'], var_name='Timepoint', value_name='Frequency')
            ax = sns.barplot(data=target_df,x="Target", y="Frequency", hue="Timepoint",alpha=0.2,ci='sd',capsize=0.1)
            sns.swarmplot(x="Target", y="Frequency", data=target_df, hue="Timepoint",dodge=True,ax=ax)
            ax.set_title(target,fontsize=20)
            ax.set_ylabel('Normalized Frequency')
            pdf.savefig(fig)
        else:
            print('Missing ' + target)
    pdf.close()
    
    
def get_hits(timepoint1,timepoint2,prefix):
    comparison = timepoint1 + "_vs_" + timepoint2
    df = pd.read_csv(prefix+'/'+ comparison + "/" +prefix+"_adj_test.txt",sep='\t')
    list_targets = df[df['p_value_corrected']<0.05]['Target'].tolist()
    list_targets.append('non-targeting')
    list_targets.append('safe-targeting')
    return list_targets
    



def u_test_method(run_prefix,PCRrepFilePath,skewFilePath,guideLib,timepoints,threshold):
    #Generate dataframes with counts for each oligo in each bin in each pcr rep 
    print('Generating first summary tables')
    create_timepoint_summary_tables(guideLib,timepoints,PCRrepFilePath,run_prefix)
    
    #Generate dataframe with counts for each oligo in each bin including skew and EC total
    print('Generating second summary tables')
    timepoints = ['D0','Pos','Neg']
    create_seq_run_summary_tables(timepoints,run_prefix,skewFilePath,guideLib)
    
    #Getting basic QCs

    # Add values to the list if they don't exist
    print('Generating basic QCs')
    timepoints = ['D0','Pos','Neg','EC_total','Skew']
    basic_qc_metrics(timepoints,run_prefix)

    #Filtering based on input threshold
    print('Filtering on minimum counts')
    timepoints = ['D0','Pos','Neg','EC_total']
    filter_dataframe(run_prefix,timepoints,threshold)
    
    #Showing correlation between halves of the PCR reps
    timepoints = ['D0','Pos','Neg']
    divider = 2
    num_iterations = 3
    plot_list = subset_correlation(num_iterations,timepoints,divider,run_prefix)
    
    #Generating log2FC with normalized values for p-value calculation for Pos vs Neg
    timepoints = ['D0','Pos','Neg','EC_total']
    
    #Generating pdf to confirm effect of normalization for Pos and Neg
    
    list_comparisons = [['Pos','Neg'],['EC_total','D0']]
    not_include_targets = []
    p_thresh = 0.05
    
    for comparison in list_comparisons:
        timepoint1 = comparison[0]
        timepoint2 = comparison[1]
        get_normalized_freqs(run_prefix,timepoints,timepoint1,timepoint2)
        confirm_normalization_qc(timepoint1,timepoint2,run_prefix)
        generate_p_values_u_test(timepoint1,timepoint2,run_prefix)
        bh_p_val_adj(run_prefix,timepoint1,timepoint2)
        show_volcano_plot(run_prefix,timepoint1,timepoint2,not_include_targets,p_thresh)
        
        
        list_targets = get_hits(timepoint1,timepoint2,run_prefix)
        df = pd.DataFrame(list_targets, columns=['Hits'])
        df.to_csv(run_prefix + '/' + timepoint1 + '_vs_' + timepoint2 + '_list_of_hits.txt',sep='\t')
        generate_target_barplot_summaries_target_specific(timepoint1,timepoint2,run_prefix,list_targets)

    #Running comparison with D0 vs skew
    u_test_D0_skew(run_prefix,p_thresh)



def generate_p_values_u_test(timepoint1,timepoint2,prefix):
    comparison = timepoint1 + "_vs_" + timepoint2
    df = pd.read_csv(prefix+'/'+comparison+"/"+prefix+"_" + timepoint1 + "_vs_" + timepoint2 + "_normalized_dataframe.txt",sep='\t')
    non_targeting_name = "non-targeting"
    safe_targeting_name = "safe-targeting"
    p_values = []
    target_list = []
    log2FC_list = []
    
    non_targeting_df = df[df['Target']==non_targeting_name]
    safe_targeting_df = df[df['Target']==safe_targeting_name]
    ctrl_dataframe = pd.concat([non_targeting_df, safe_targeting_df], ignore_index=True)
    controls = ctrl_dataframe["log2FC_"+comparison+"_D0_ctrl_norm"]
    

    # Iterate over the targets and perform a two-sample t-test
    for target in df['Target'].unique():    
        targeting = df.loc[df['Target'] == target]["log2FC_"+comparison+"_D0_ctrl_norm"]
        u_stat, p_value = mannwhitneyu(controls, targeting)
        p_values.append(p_value)
        target_list.append(target)
        log2FC_list.append(targeting.mean())
    
    combined_target_significance = pd.DataFrame({'Target':target_list,'p_value':p_values,'log2FC':log2FC_list})
        
    combined_target_significance.set_index('Target').to_csv(prefix+'/'+ comparison + "/" + prefix+"_"+comparison+"_unadj_t_test.txt",sep='\t')

def t_test_method(run_prefix,PCRrepFilePath,skewFilePath,guideLib,timepoints,threshold):
    #Generate dataframes with counts for each oligo in each bin in each pcr rep 
    print('Generating first summary tables')
    create_timepoint_summary_tables(guideLib,timepoints,PCRrepFilePath,run_prefix)
    
    #Generate dataframe with counts for each oligo in each bin including skew and EC total
    print('Generating second summary tables')
    timepoints = ['D0','Pos','Neg']
    create_seq_run_summary_tables(timepoints,run_prefix,skewFilePath,guideLib)
    
    #Getting basic QCs

    # Add values to the list if they don't exist
    print('Generating basic QCs')
    timepoints = ['D0','Pos','Neg','EC_total','Skew']
    basic_qc_metrics(timepoints,run_prefix)

    #Filtering based on input threshold
    print('Filtering on minimum counts')
    timepoints = ['D0','Pos','Neg','EC_total']
    filter_dataframe(run_prefix,timepoints,threshold)
    
    #Showing correlation between halves of the PCR reps
    timepoints = ['D0','Pos','Neg']
    divider = 2
    num_iterations = 3
    plot_list = subset_correlation(num_iterations,timepoints,divider,run_prefix)
    
    #Generating log2FC with normalized values for p-value calculation for Pos vs Neg
    timepoints = ['D0','Pos','Neg','EC_total']
    
    #Generating pdf to confirm effect of normalization for Pos and Neg
    
    list_comparisons = [['Pos','Neg'],['EC_total','D0']]
    not_include_targets = []
    p_thresh = 0.05
    
    
    for comparison in list_comparisons:
        timepoint1 = comparison[0]
        timepoint2 = comparison[1]
        get_normalized_freqs(run_prefix,timepoints,timepoint1,timepoint2)
        confirm_normalization_qc(timepoint1,timepoint2,run_prefix)
        generate_p_values_t_test(timepoint1,timepoint2,run_prefix)
        bh_p_val_adj(run_prefix,timepoint1,timepoint2)
        show_volcano_plot(run_prefix,timepoint1,timepoint2,not_include_targets,p_thresh)
        
        
        list_targets = get_hits(timepoint1,timepoint2,run_prefix)
        df = pd.DataFrame(list_targets, columns=['Hits'])
        df.to_csv(run_prefix + '/' + timepoint1 + '_vs_' + timepoint2 + '_list_of_hits.txt',sep='\t')        
        generate_target_barplot_summaries_target_specific(timepoint1,timepoint2,run_prefix,list_targets)

    #Running comparison with D0 vs skew
    t_test_D0_skew(run_prefix,p_thresh)



def get_normalized_freqs(prefix,timepoints,timepoint1,timepoint2):
    input_df = pd.read_csv(prefix+"/"+prefix+"_filtered_total_counts.txt",sep='\t')
    non_targeting_name = 'non-targeting'
    safe_targeting_name = 'safe-targeting'
    
    comparison = timepoint1 + '_vs_' + timepoint2

    #Calculating frequency in bin
    for timepoint in timepoints:
        total_count = input_df[timepoint].sum()
        input_df[timepoint + "_freq"] = input_df[timepoint]/total_count

    #Normalize gRNA frequency by D0 frequency 
    for timepoint in timepoints:
        input_df[timepoint + "_D0_norm"] = input_df[timepoint+"_freq"]/input_df['D0_freq']

    #Normalize gRNA frequency by safe-targeting and non-targeting average
    non_targeting_df = input_df[input_df['Target']==non_targeting_name]
    safe_targeting_df = input_df[input_df['Target']==safe_targeting_name]
    ctrl_dataframe = pd.concat([non_targeting_df, safe_targeting_df], ignore_index=True)
    ctrl_avg = ctrl_dataframe[timepoint+"_D0_norm"].mean()
    for timepoint in timepoints:
        input_df[timepoint+"_D0_ctrl_norm"] = input_df[timepoint+"_D0_norm"]/ctrl_avg


    #Get fold change between bins of interest without normalization
    input_df[comparison] = input_df[timepoint1+"_freq"]/input_df[timepoint2+"_freq"]

    #Get log2_FC without normalization
    input_df["log2FC_"+comparison] = np.log2(input_df[comparison])

    #Get fold change between bins of interest with D0 normalization
    input_df[comparison+"_D0_norm"] = input_df[timepoint1+"_D0_norm"]/input_df[timepoint2+"_D0_norm"]

    #Get log2_FC with D0 normalization
    input_df["log2FC_"+comparison+"_D0_norm"] = np.log2(input_df[comparison+"_D0_norm"])

    #Get fold change between bins of interest with normalization
    input_df[comparison+"_D0_ctrl_norm"] = input_df[timepoint1+"_D0_ctrl_norm"]/input_df[timepoint2+"_D0_ctrl_norm"]

    #Get log2_FC with normalization
    input_df["log2FC_"+comparison+"_D0_ctrl_norm"] = np.log2(input_df[comparison+"_D0_ctrl_norm"])

    
    
    parent_directory_path = '/Users/cjmunger/Documents/230523_Pipeline'
    parent_directory_path = parent_directory_path + '/' + prefix + '/'
    # Construct the complete path for the new folder
    new_folder_path = os.path.join(parent_directory_path, comparison)
    create_folder_if_not_exists(new_folder_path)
    input_df.set_index('OligoID').to_csv(prefix+'/'+comparison+"/"+prefix+"_" + timepoint1 + "_vs_" + timepoint2 + "_normalized_dataframe.txt",sep='\t')

    
    
    
    
def create_seq_run_summary_tables(timepoints,run_prefix,skewFilePath,guidelib_path):
    #Setting up summary table
    total_df = pd.read_csv(guidelib_path,sep='\t')
    total_df = total_df[['OligoID','Target']]
    
    #Creating columns for timepoint summaries (adding all PCR rep counts)
    for timepoint in timepoints:
        timepoint_df = pd.read_csv(run_prefix+"/"+run_prefix+'_'+timepoint+"_total.txt",sep='\t')
        timepoint_df[timepoint] = timepoint_df.drop(['OligoID', 'Target'], axis=1).sum(axis=1)
        total_df = pd.merge(total_df, timepoint_df[['OligoID',timepoint]], on='OligoID')
    
    #Including EC_total counts    
    total_df['EC_total'] = total_df['Pos'] + total_df['Neg']

    #Including skew counts    
    skew_df = pd.read_csv(skewFilePath,sep='\t')
    column_mapping = {'counts': 'Skew', 'OligoID': 'OligoID'}
    skew_df = skew_df.rename(columns=column_mapping)
    total_df = pd.merge(total_df, skew_df[['OligoID','Skew']], on='OligoID')
    
    
    
    total_df.to_csv(run_prefix+"/"+run_prefix+'_total_counts.txt',sep='\t',index=False)
    
    
def create_timepoint_summary_tables(guidelib_path,timepoints,pcrrep_folder,prefix):
    guidelib_df = pd.read_csv(guidelib_path,sep='\t')
    for timepoint in timepoints:
        timepoint_combined_df = guidelib_df[['OligoID','Target']]
        reps = get_max_pcr_rep(pcrrep_folder,timepoint)
        for rep in range(reps):
            rep=rep+1
            file_name = pcrrep_folder+'/byPCRRep/MACS-Rep1-1-PCR' + str(rep) + ".bin_counts.txt"
            temp_df = pd.read_csv(file_name,sep='\t')
            if timepoint in temp_df.columns:
                temp_df = temp_df[['OligoID',timepoint]]
                temp_df.columns=['OligoID',timepoint+"_"+str(rep)]
                if temp_df[timepoint+"_"+str(rep)].sum()>0:
                    timepoint_combined_df = pd.merge(timepoint_combined_df,temp_df,on=['OligoID'],how='outer')
        
        timepoint_combined_df.fillna(0,inplace=True)
        
        create_folder_if_not_exists(prefix)
        timepoint_combined_df = timepoint_combined_df.merge(guidelib_df[['OligoID','Target']],on='OligoID')
        
        column_mapping = {'Target_x': 'Target'}
        
        timepoint_combined_df = timepoint_combined_df.rename(columns=column_mapping)     
        timepoint_combined_df = timepoint_combined_df.drop('Target_y', axis=1)
        timepoint_combined_df.set_index(['OligoID']).to_csv(prefix+"/"+prefix+'_'+timepoint+"_total.txt",sep='\t')

        
def get_max_pcr_rep(filepath,timepoint):
    # Get the list of files in the folder
    file_list = os.listdir(filepath+"/counts")
    num = 0
    # Print the list of files
    for file in file_list:
        if timepoint in file:
            start_index = file.find("_") + 1
            end_index = file.find(".")
            test_num = int(file[start_index:end_index])
            if test_num > num:
                num = test_num
    return num


def create_folder_if_not_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

        
        

def basic_qc_metrics(timepoints,save_name_prefix):
    
    df = pd.read_csv(save_name_prefix+"/"+save_name_prefix+'_total_counts.txt',sep='\t')
    # Create a PDF file to save the histograms
    pdf_file = save_name_prefix+'/'+save_name_prefix+"_basic_histogram_qcs.pdf"

    # Create a PDF pages object
    pdf_pages = PdfPages(pdf_file)
    for timepoint in timepoints:
        #Show distribution of sgRNA counts
        fig, ax = plt.subplots()
        ax.hist(df[timepoint], bins=50)
        ax.set_xlabel('Counts in ' + timepoint)
        ax.set_ylabel('# sgRNA')
        ax.set_title('Histogram of ' + timepoint + ' sgRNA counts (' + save_name_prefix + ')')


        #Display median and mean and what % have more than 100 counts
        median = np.median(df[timepoint])
        plt.axvline(median, color='red')
        plt.text(plt.xlim()[1]*.5, plt.ylim()[1]*0.9, f'Median = {median:.1f}', ha='center', va='top', color='red', transform=plt.gca().transData)

        mean = np.mean(df[timepoint])
        plt.axvline(mean, color='blue')
        plt.text(plt.xlim()[1]*.5, plt.ylim()[1]*0.8, f'Mean = {mean:.1f}', ha='center', va='top', color='blue', transform=plt.gca().transData)

        percentage = (df[timepoint] > 100).sum() / len(df) * 100
        # Display the percentage

        plt.text(plt.xlim()[1]*.5, plt.ylim()[1]*0.7, f'% >100 = {percentage:.1f}', ha='center', va='top', color='black', transform=plt.gca().transData)

        save_name = save_name_prefix + "_" + timepoint+"_sgRNA_distribution"
        #plt.savefig(save_name+'.pdf',bbox_inches='tight')

        pdf_pages.savefig()
        plt.close()
        
        
    for timepoint in timepoints:
        #Group sgRNAs by target and sum counts dropping the non-targeting and safe-targeting totals (since they are much larger)
        sums = df.groupby('Target')[timepoint].sum()
        sums = sums.sort_values(ascending=False).iloc[2:]
        plt.hist(sums, bins=100)

        # Set the title and axis labels
        plt.title('Histogram of ' + timepoint + ' Target Counts (' + save_name_prefix + ')')
        plt.xlabel('Target Counts in ' + timepoint)
        plt.ylabel('# of Targets')


        #Display median and mean
        median = sums.median()
        plt.axvline(median, color='red')
        plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.9, f'Median = {median:.1f}', ha='center', va='top', color='red', transform=plt.gca().transData)

        mean = sums.mean()
        plt.axvline(mean, color='blue')
        plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.8, f'Mean = {mean:.1f}', ha='center', va='top', color='blue', transform=plt.gca().transData)

        save_name = save_name_prefix + "_" + timepoint+"_target_distribution"
        #plt.savefig(save_name+'.pdf',bbox_inches='tight')

        # Display the plot
        pdf_pages.savefig()
        plt.close()
        

    #Generate PCR rep correlation plots 
        
    
    pdf_pages.close()

    
    
def filter_dataframe(prefix,timepoints,min_count):
    file_path = prefix+"/" + prefix + "_filter_metrics.txt"
    df = pd.read_csv(prefix+"/"+prefix+'_total_counts.txt',sep='\t')
    rows_before = len(df)
    targets_before = len(df['Target'].unique())
    summary_text = ""
    
    

    
    
    #Total counts prior to filtering
    summary_text = summary_text + 'Before filtering for minimum count (' + str(min_count) + ')\n'
    for timepoint in timepoints:
        coverage = df[timepoint].sum()/len(df)
        summary_text = summary_text + timepoint + ' total counts: ' + str(df[timepoint].sum()) + f' (Coverage = {coverage:.1f}x)\n'
    summary_text = summary_text + '\n'

    #Calculating how many sgRNAs are totally absent from a bin
    for timepoint in timepoints:
        temp_df = df[df[timepoint]==0]
        summary_text = summary_text + str(len(temp_df))+ ' sgRNAs missing from ' + timepoint + '\n'
        

    # % of sgRNA with < min_counts in each bin
    for timepoint in timepoints:
        percent_lower = len(df[df[timepoint]<min_count])/len(df)*100
        percent_lower = f'{percent_lower:.1f}'
        summary_text = summary_text + timepoint + ' bin: ' + percent_lower + "% <" + str(min_count) + ' counts\n'


    #Dropping by minimum count in D0
    df = df[df['D0']>=min_count]

    summary_text = summary_text + '\n'

    #Total counts after to filtering
    for timepoint in timepoints:
        coverage = df[timepoint].sum()/len(df)
        summary_text = summary_text + timepoint + ' total counts: ' + str(df[timepoint].sum()) + f' (Average Remaining Coverage = {coverage:.1f}x)\n'

        
    #Showing how sgRNAs are affected from filtering
    rows_after = len(df)
    lost_rows = rows_before - rows_after
    percent_sgRNA_lost = lost_rows/rows_before*100
    summary_text = summary_text + f'\nLost {lost_rows:.0f} sgRNAs from filtering ({percent_sgRNA_lost:.1f}% of total)\n'

    
    #Showing how targets are affected from filtering
    targets_after = len(df['Target'].unique())
    lost_targets = targets_before - targets_after
    percent_target_lost = lost_targets/targets_before*100
    summary_text = summary_text + f'Lost {lost_targets:.0f} targets from filtering ({percent_target_lost:.1f}% of total)\n'


    with open(file_path, 'w') as file:
        file.write(summary_text)
        
    #Adding 1 to prevent 0 count bins from acting strange
    for timepoint in timepoints:
        df[timepoint] = df[timepoint]+1
    
    df.set_index('OligoID').to_csv(prefix+"/"+prefix+"_filtered_total_counts.txt",sep='\t')


    
def subset_correlation(num_iterations,timepoints,divider,prefix):
    # Create a PDF file to save the histograms
    pdf_file = prefix + "/"+prefix+"_subset_correlation.pdf"

    # Create a PDF pages object
    pdf_pages = PdfPages(pdf_file)

    for timepoint in timepoints:
        corr_list = []
        df = pd.read_csv(prefix+'/' + prefix + "_" + timepoint+'_total.txt',sep='\t')
        for i in range(num_iterations):
            print(i)

            df1, df2 = split_dataframe(df,divider)

            # sum the counts across rows for each dataframe while keeping the 'oligoid' and 'target' columns
            df1['sum'] = df1.sum(axis=1)
            df2['sum'] = df2.sum(axis=1)

            # get the correlation between the 'sum' column of each dataframe and append it to the list
            corr = df1['sum'].corr(df2['sum'])
            corr_list.append(corr)
            
        #Plot range of correlations
        mean = np.mean(corr_list)
        std = np.std(corr_list)
        plt.bar(0, mean, yerr=std, capsize=10,alpha=0.5)
        plt.scatter(np.zeros_like(corr_list), corr_list, color="red", s=10,alpha=0.5)
        plt.xticks([], [])
        plt.xlabel(timepoint)
        plt.ylabel('Correlation')
        plt.title(timepoint + ' correlation when splitting PCR reps (' + prefix + ')')
        pdf_pages.savefig()
        plt.show()
        plt.close()
            
        #Plotting last of the correlation groupings
        x = df1['sum']
        y = df2['sum']
        
        correlation = x.corr(y)

        plt.scatter(x, y)
        plt.xlabel('First Half PCR Reps')
        plt.ylabel('Second Half PCR Reps')
        plt.title(timepoint + ' sgRNA counts (' + prefix + ')')
        plt.text(0.5, 0.9, f'Correlation: {correlation:.2f}', ha='center', va='center', transform=plt.gca().transAxes,color='red')
        
        pdf_pages.savefig()
        plt.show()

        plt.close()
        
        #Log10 transforming after +1
        
        x = np.log10(df1['sum']+1)
        y = np.log10(df2['sum']+1)
        
        correlation = x.corr(y)

        plt.scatter(x, y)
        plt.xlabel('Rep 1')
        plt.ylabel('Rep 2')
        plt.title(timepoint + ' sgRNA counts, log10 (' + prefix + ')')
        plt.text(0.5, 0.9, f'Correlation: {correlation:.2f}', ha='center', va='center', transform=plt.gca().transAxes,color='red')
        
        
        
        pdf_pages.savefig(dpi=50)
        plt.show()

        plt.close()

        # show figure of the correlation values
        xlabel = "Correlation"
        ylabel = timepoint + " Correlation"
        title = timepoint + "_" + str(num_iterations)+'_samples_correlations'
        
        mean = np.mean(corr_list)
        std = np.std(corr_list)
        plt.bar(0, mean, yerr=std, capsize=10,alpha=0.5)
        plt.scatter(np.zeros_like(corr_list), corr_list, color="red", s=10,alpha=0.5)
        plt.xticks([], [])
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.title(title)

        pdf_pages.savefig(dpi=50)
        plt.close()
        
        #Getting same graphs by grouping by target
        
        #df = df.groupby('Target').agg({'A': 'sum', 'B': 'sum', 'C': 'sum'})
        #df = df.reset_index().drop('OligoID', axis=1)


        
    pdf_pages.close()


def split_dataframe(df,divider):
    # create two empty dataframes with 'target' and 'oligoid' columns
    df1 = pd.DataFrame(columns=['Target', 'OligoID'])
    df2 = pd.DataFrame(columns=['Target', 'OligoID'])
    
    # create a list of column indices and shuffle it randomly
    col_indices = list(range(len(df.columns)))
    random.shuffle(col_indices)

    # loop through the shuffled column indices and assign them to one of the new dataframes
    for i, col_idx in enumerate(col_indices):
        col_name = df.columns[col_idx]
        if i % divider == 0:
            df1[col_name] = df[col_name]
        else:
            df2[col_name] = df[col_name]
    
    # return the two new dataframes
    return df1, df2




def confirm_normalization_qc(timepoint1,timepoint2,prefix):
    comparison_base = timepoint1 + '_vs_' + timepoint2
    comparison = comparison_base
    # Create a PDF file to save the histograms
    pdf_file = prefix+'/'+comparison + "/"+prefix+"_log2FC_histograms.pdf"

    # Create a PDF pages object
    pdf_pages = PdfPages(pdf_file)
    df = pd.read_csv(prefix+'/'+comparison+"/"+prefix+"_" + timepoint1 + "_vs_" + timepoint2 + "_normalized_dataframe.txt",sep='\t')
    non_targeting_name = 'non-targeting'
    safe_targeting_name = 'safe-targeting'
    num_bins = 100
    
    


    ####Plotting  without normalization 

    #Making dataframes of just safe-targeting and non-targeting sgRNAs
    non_target_df = df[df['Target']==non_targeting_name]
    safe_target_df = df[df['Target']==safe_targeting_name]
    

    #Showing distribution of all log2FC without normalization
    fig, ax = plt.subplots()
    ax.hist(df["log2FC_"+comparison], bins=num_bins)
    ax.set_xlabel('log2FC')
    ax.set_ylabel('Frequency')
    ax.set_title('(All Filtered Oligos) Histogram \n log2FC W/O Normalization (' + comparison + ")")

    median = df["log2FC_"+comparison].median()
    plt.axvline(median, color='red')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.9, f'Median = {median:.2f}', ha='center', va='top', color='red', transform=plt.gca().transData)

    mean = df["log2FC_"+comparison].mean()
    plt.axvline(mean, color='blue')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.8, f'Mean = {mean:.2f}', ha='center', va='top', color='blue', transform=plt.gca().transData)


    pdf_pages.savefig()
    plt.close()


    #Showing non-targeting distribution of log2FC without normalization
    fig, ax = plt.subplots()
    ax.hist(non_target_df["log2FC_"+comparison], bins=num_bins)
    ax.set_xlabel('log2FC')
    ax.set_ylabel('Frequency')
    ax.set_title('(Non-Targeting) Histogram \n log2FC W/O Normalization (' + comparison + ")")

    median = non_target_df["log2FC_"+comparison].median()
    plt.axvline(median, color='red')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.9, f'Median = {median:.2f}', ha='center', va='top', color='red', transform=plt.gca().transData)

    mean = non_target_df["log2FC_"+comparison].mean()
    plt.axvline(mean, color='blue')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.8, f'Mean = {mean:.2f}', ha='center', va='top', color='blue', transform=plt.gca().transData)

    pdf_pages.savefig()
    plt.close()


    #Showing safe-targeting distribution of log2FC without normalization
    fig, ax = plt.subplots()
    ax.hist(safe_target_df["log2FC_"+comparison], bins=num_bins)
    ax.set_xlabel('log2FC')
    ax.set_ylabel('Frequency')
    ax.set_title('(Safe-Targeting) Histogram \n log2FC W/O Normalization (' + comparison + ")")

    median = safe_target_df["log2FC_"+comparison].median()
    plt.axvline(median, color='red')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.9, f'Median = {median:.2f}', ha='center', va='top', color='red', transform=plt.gca().transData)

    mean = safe_target_df["log2FC_"+comparison].mean()
    plt.axvline(mean, color='blue')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.8, f'Mean = {mean:.2f}', ha='center', va='top', color='blue', transform=plt.gca().transData)
    
    pdf_pages.savefig()
    plt.close()




    ####Plotting  with D0 (not ctrl) normalization 

    comparison = comparison_base + "_D0_norm"



    #Showing distribution of all log2FC with normalization
    fig, ax = plt.subplots()
    ax.hist(df["log2FC_"+comparison], bins=num_bins)
    ax.set_xlabel('log2FC')
    ax.set_ylabel('Frequency')
    ax.set_title('(All Filtered Oligos) Histogram \n log2FC W/ D0 Normalization (' + comparison + ")")

    median = df["log2FC_"+comparison].median()
    plt.axvline(median, color='red')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.9, f'Median = {median:.2f}', ha='center', va='top', color='red', transform=plt.gca().transData)

    mean = df["log2FC_"+comparison].mean()
    plt.axvline(mean, color='blue')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.8, f'Mean = {mean:.2f}', ha='center', va='top', color='blue', transform=plt.gca().transData)

    pdf_pages.savefig()
    plt.close()


    #Showing non-targeting distribution of log2FC with normalization
    fig, ax = plt.subplots()
    ax.hist(non_target_df["log2FC_"+comparison], bins=num_bins)
    ax.set_xlabel('log2FC')
    ax.set_ylabel('Frequency')
    ax.set_title('(Non-Targeting) Histogram \n log2FC W/ D0 Normalization (' + comparison + ")")

    median = non_target_df["log2FC_"+comparison].median()
    plt.axvline(median, color='red')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.9, f'Median = {median:.2f}', ha='center', va='top', color='red', transform=plt.gca().transData)

    mean = non_target_df["log2FC_"+comparison].mean()
    plt.axvline(mean, color='blue')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.8, f'Mean = {mean:.2f}', ha='center', va='top', color='blue', transform=plt.gca().transData)

    pdf_pages.savefig()
    plt.close()


    #Showing safe-targeting distribution of log2FC with normalization
    fig, ax = plt.subplots()
    ax.hist(safe_target_df["log2FC_"+comparison], bins=num_bins)
    ax.set_xlabel('log2FC')
    ax.set_ylabel('Frequency')
    ax.set_title('(Safe-Targeting) Histogram \n log2FC W/ D0 Normalization (' + comparison + ")")

    median = safe_target_df["log2FC_"+comparison].median()
    plt.axvline(median, color='red')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.9, f'Median = {median:.2f}', ha='center', va='top', color='red', transform=plt.gca().transData)

    mean = safe_target_df["log2FC_"+comparison].mean()
    plt.axvline(mean, color='blue')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.8, f'Mean = {mean:.2f}', ha='center', va='top', color='blue', transform=plt.gca().transData)

    pdf_pages.savefig()
    plt.close()
    
    
    ####Plotting  with D0 and ctrl normalization 

    comparison = comparison_base + "_D0_ctrl_norm"



    #Showing distribution of all log2FC with normalization
    fig, ax = plt.subplots()
    ax.hist(df["log2FC_"+comparison], bins=num_bins)
    ax.set_xlabel('log2FC')
    ax.set_ylabel('Frequency')
    ax.set_title('(All Filtered Oligos) Histogram \n log2FC W/ D0 + ctrl Normalization (' + comparison + ")")

    median = df["log2FC_"+comparison].median()
    plt.axvline(median, color='red')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.9, f'Median = {median:.2f}', ha='center', va='top', color='red', transform=plt.gca().transData)

    mean = df["log2FC_"+comparison].mean()
    plt.axvline(mean, color='blue')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.8, f'Mean = {mean:.2f}', ha='center', va='top', color='blue', transform=plt.gca().transData)

    pdf_pages.savefig()
    plt.close()


    #Showing non-targeting distribution of log2FC with normalization
    fig, ax = plt.subplots()
    ax.hist(non_target_df["log2FC_"+comparison], bins=num_bins)
    ax.set_xlabel('log2FC')
    ax.set_ylabel('Frequency')
    ax.set_title('(Non-Targeting) Histogram \n log2FC W/ D0 + ctrl Normalization (' + comparison + ")")

    median = non_target_df["log2FC_"+comparison].median()
    plt.axvline(median, color='red')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.9, f'Median = {median:.2f}', ha='center', va='top', color='red', transform=plt.gca().transData)

    mean = non_target_df["log2FC_"+comparison].mean()
    plt.axvline(mean, color='blue')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.8, f'Mean = {mean:.2f}', ha='center', va='top', color='blue', transform=plt.gca().transData)

    pdf_pages.savefig()
    plt.close()


    #Showing safe-targeting distribution of log2FC with normalization
    fig, ax = plt.subplots()
    ax.hist(safe_target_df["log2FC_"+comparison], bins=num_bins)
    ax.set_xlabel('log2FC')
    ax.set_ylabel('Frequency')
    ax.set_title('(Safe-Targeting) Histogram \n log2FC W/ D0 + ctrl Normalization (' + comparison + ")")

    median = safe_target_df["log2FC_"+comparison].median()
    plt.axvline(median, color='red')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.9, f'Median = {median:.2f}', ha='center', va='top', color='red', transform=plt.gca().transData)

    mean = safe_target_df["log2FC_"+comparison].mean()
    plt.axvline(mean, color='blue')
    plt.text(plt.xlim()[1]*.8, plt.ylim()[1]*0.8, f'Mean = {mean:.2f}', ha='center', va='top', color='blue', transform=plt.gca().transData)

    pdf_pages.savefig()
    plt.close()
    
    pdf_pages.close()

def generate_p_values_t_test(timepoint1,timepoint2,prefix):
    comparison = timepoint1 + "_vs_" + timepoint2
    df = pd.read_csv(prefix+'/'+comparison+"/"+prefix+"_" + timepoint1 + "_vs_" + timepoint2 + "_normalized_dataframe.txt",sep='\t')
    non_targeting_name = "non-targeting"
    safe_targeting_name = "safe-targeting"
    p_values = []
    target_list = []
    log2FC_list = []
    
    non_targeting_df = df[df['Target']==non_targeting_name]
    safe_targeting_df = df[df['Target']==safe_targeting_name]
    ctrl_dataframe = pd.concat([non_targeting_df, safe_targeting_df], ignore_index=True)
    controls = ctrl_dataframe["log2FC_"+comparison+"_D0_ctrl_norm"]
    

    # Iterate over the targets and perform a two-sample t-test
    for target in df['Target'].unique():    
        targeting = df.loc[df['Target'] == target]["log2FC_"+comparison+"_D0_ctrl_norm"]
        t_stat, p_value = ttest_ind(controls, targeting, equal_var=False)
        p_values.append(p_value)
        target_list.append(target)
        log2FC_list.append(targeting.mean())
    
    combined_target_significance = pd.DataFrame({'Target':target_list,'p_value':p_values,'log2FC':log2FC_list})
        
    combined_target_significance.set_index('Target').to_csv(prefix+'/'+ comparison + "/" + prefix+"_"+comparison+"_unadj_t_test.txt",sep='\t')

def bh_p_val_adj(prefix,timepoint1,timepoint2):
    comparison = timepoint1 + "_vs_" + timepoint2
    df = pd.read_csv(prefix+'/'+ comparison + "/" + prefix + "_" + comparison  + "_unadj_t_test.txt",sep='\t')
    file_path = prefix+'/'+ comparison + "/" +prefix+ "_volcano_metrics.txt"
    summary_text = ""
    

    
    len_pre_filter = len(df)
    #Dropping those that didn't have p-values
    df.dropna(inplace=True)
    len_post_filter = len(df)
    diff = len_pre_filter - len_post_filter
    summary_text = summary_text + 'Lost ' + str(diff) + ' targets due to <2 oligos getting to this point\n'
    
    #Performing a Benjamini-Hochberg correction 
    _, pvals_corrected, _, _ = multipletests(df['p_value'], method='fdr_bh')
    # add the corrected p-values to the DataFrame
    df['p_value_corrected'] = pvals_corrected
    hit_df = df[df['p_value_corrected']<0.05]

    summary_text = summary_text + str(len(hit_df))+" targets with adjusted p-value <0.05\n"

    summary_text = summary_text + str(len(hit_df[hit_df['log2FC']<0]))+" are negative hits\n"
    summary_text = summary_text + str(len(hit_df[hit_df['log2FC']>0]))+" are positive hits\n"


    with open(file_path, 'w') as file:
        file.write(summary_text)
    
    df.set_index('Target').to_csv(prefix+'/'+ comparison + "/" +prefix+"_adj_test.txt",sep='\t')


def show_volcano_plot(prefix,timepoint1,timepoint2,not_include_targets,p_thresh):
    comparison = timepoint1 + "_vs_" + timepoint2
    df = pd.read_csv(prefix+'/'+ comparison + "/" +prefix+"_adj_test.txt",sep='\t')
    volcano_title = prefix + "/" + comparison + "/volcano_" + prefix + "_" + timepoint1 + "_vs_" + timepoint2

    try:
        for target in not_include_targets:
            df = df[df['Target']!=target]
        targets_to_include=get_deg_genes(p_thresh,df)
        targets_to_include.append('non-targeting')
        targets_to_include.append('safe-targeting')
        #If you want to save the image, use the bottom command
        #visuz.GeneExpression.volcano(df=df,lfc='log2FC',pv='p_value_corrected',lfc_thr=(0,0),pv_thr=(p_thresh,p_thresh),show=True,geneid='Target',genenames=tuple(targets_to_include),plotlegend=True,legendpos='upper right',legendanchor=(1.46,1),dim=(10,10),gfont=20,sign_line=True,figname=volcano_title)
        visuz.GeneExpression.volcano(df=df,lfc='log2FC',pv='p_value_corrected',lfc_thr=(0,0),pv_thr=(p_thresh,p_thresh),geneid='Target',genenames=tuple(targets_to_include),plotlegend=True,legendpos='upper right',legendanchor=(1.46,1),dim=(10,10),gfont=20,sign_line=True,figname=volcano_title)
    except:
        print('Either/or positive or negative regulators lack stasticially significant result with p_value threshold of 0.05')
        
def get_deg_genes(pv_thr,df):
    deg_gene_list = []
    for index, row in df.iterrows():
        if row['p_value_corrected']<pv_thr:
            deg_gene_list.append(row['Target'])
    return deg_gene_list    

def generate_target_barplot_summaries(timepoint1,timepoint2,prefix):
    comparison = timepoint1 + "_vs_" + timepoint2
    pdf_title = prefix+'/'+comparison+"/"+prefix+"_" + timepoint1 + "_vs_" + timepoint2 + "_target_freq_swarms.pdf"
    pdf = matplotlib.backends.backend_pdf.PdfPages(pdf_title)  
    
    if timepoint2 != 'Skew':
        df = pd.read_csv(prefix+'/'+comparison+"/"+prefix+"_" + timepoint1 + "_vs_" + timepoint2 + "_normalized_dataframe.txt",sep='\t')
    else:
        df = pd.read_csv(prefix+'/'+comparison+"/"+prefix+"_" + timepoint1 + "_vs_" + timepoint2 + "_normalized_dataframe.txt",sep='\t')
    common_suffix = '_D0_ctrl_norm'

    for target in df['Target'].unique().tolist():
        print(target)
        a4_dims = (20, 8.27)
        fig, ax = plt.subplots(figsize=a4_dims)
        target_df = df[df['Target']==target][['OligoID','Target','D0'+common_suffix,'Pos'+common_suffix,'Neg'+common_suffix,'EC_total'+common_suffix]]
        column_mapping = {'OligoID': 'OligoID', 'Target': 'Target', 'D0'+common_suffix: 'D0', 'Pos'+common_suffix: 'Pos', 'Neg'+common_suffix: 'Neg','EC_total'+common_suffix: 'EC_total',}
        target_df = target_df.rename(columns=column_mapping)

        target_df = target_df.melt(id_vars=['OligoID', 'Target'], var_name='Timepoint', value_name='Frequency')


        ax = sns.barplot(data=target_df,x="Target", y="Frequency", hue="Timepoint",alpha=0.2,ci='sd',capsize=0.1)
        sns.swarmplot(x="Target", y="Frequency", data=target_df, hue="Timepoint",dodge=True,ax=ax)
        ax.set_title(target,fontsize=20)
        ax.set_ylabel('Normalized Frequency')
        pdf.savefig(fig)
    pdf.close()

    
#Compare skew to D0

def compare_skew_D0_limited_filter(prefix,timepoints):
    df = pd.read_csv(prefix + "/" + prefix + "_total_counts.txt",sep='\t')
    #Calculating frequency in bin
    for timepoint in timepoints:
        total_count = df[timepoint].sum()
        df[timepoint + "_freq"] = df[timepoint]/total_count
 

In [ ]:

def t_test_D0_skew(run_prefix,threshold):
    timepoint1 = 'D0'
    timepoint2 = 'Skew'    
    comparison = timepoint1 + "_vs_" + timepoint2
    timepoints = ['D0','Skew']
    not_include_targets = []
    p_thresh = 0.05

    parent_directory_path = '/Users/cjmunger/Documents/230523_Pipeline'
    parent_directory_path = parent_directory_path + '/' + run_prefix + '/'
    # Construct the complete path for the new folder
    new_folder_path = os.path.join(parent_directory_path, comparison)
    create_folder_if_not_exists(new_folder_path)
    
    
    
    
    
    df = pd.read_csv(prefix + "/" + prefix + "_total_counts.txt",sep='\t')
    #Calculating frequency in bin and drop those with a value below a threshold 

    for timepoint in timepoints:
        df = df[df[timepoint]>threshold]
        total_count = df[timepoint].sum()
        df[timepoint + "_freq"] = df[timepoint]/total_count


    #Calculate log2FC between the bins 
    #Get fold change between bins of interest without normalization
    df[comparison] = df[timepoint1+"_freq"]/df[timepoint2+"_freq"]

    #Get log2_FC without normalization
    df["log2FC_"+comparison] = np.log2(df[comparison])
    df.set_index('OligoID').to_csv(prefix+"/"+comparison + "/" + prefix+"_D0_skew_comparison.txt",sep='\t')





    df = pd.read_csv(run_prefix+"/"+comparison+"/"+run_prefix+"_D0_skew_comparison.txt",sep='\t')
    non_targeting_name = "non-targeting"
    safe_targeting_name = "safe-targeting"
    p_values = []
    target_list = []
    log2FC_list = []
    
    non_targeting_df = df[df['Target']==non_targeting_name]
    safe_targeting_df = df[df['Target']==safe_targeting_name]
    ctrl_dataframe = pd.concat([non_targeting_df, safe_targeting_df], ignore_index=True)
    controls = ctrl_dataframe["log2FC_D0_vs_Skew"]
    

    # Iterate over the targets and perform a two-sample t-test
    for target in df['Target'].unique():    
        targeting = df.loc[df['Target'] == target]["log2FC_D0_vs_Skew"]
        t_stat, p_value = ttest_ind(controls, targeting, equal_var=False)
        p_values.append(p_value)
        target_list.append(target)
        log2FC_list.append(targeting.mean())
    
    combined_target_significance = pd.DataFrame({'Target':target_list,'p_value':p_values,'log2FC':log2FC_list})
        
    combined_target_significance.set_index('Target').to_csv(run_prefix+ "/" +comparison+"/"+run_prefix+"_"+comparison+"_unadj_t_test.txt",sep='\t')

    

    bh_p_val_adj(run_prefix,timepoint1,timepoint2)

    show_volcano_plot(run_prefix,timepoint1,timepoint2,not_include_targets,p_thresh)
    
    list_targets = get_hits(timepoint1,timepoint2,run_prefix)
    df = pd.DataFrame(list_targets, columns=['Hits'])
    df.to_csv(run_prefix + '/' + timepoint1 + '_vs_' + timepoint2 + '_list_of_hits.txt',sep='\t')
    
    generate_target_barplot_summaries_target_specific_skew(timepoint1,timepoint2,run_prefix,list_targets)
    

def u_test_D0_skew(run_prefix,threshold):
    timepoint1 = 'D0'
    timepoint2 = 'Skew'    
    comparison = timepoint1 + "_vs_" + timepoint2
    timepoints = ['D0','Skew']
    not_include_targets = []
    p_thresh = 0.05

    parent_directory_path = '/Users/cjmunger/Documents/230523_Pipeline'
    parent_directory_path = parent_directory_path + '/' + run_prefix + '/'
    # Construct the complete path for the new folder
    new_folder_path = os.path.join(parent_directory_path, comparison)
    create_folder_if_not_exists(new_folder_path)
    
    
    
    
    
    df = pd.read_csv(prefix + "/" + prefix + "_total_counts.txt",sep='\t')
    #Calculating frequency in bin and drop those with a value below a threshold 

    for timepoint in timepoints:
        df = df[df[timepoint]>threshold]
        total_count = df[timepoint].sum()
        df[timepoint + "_freq"] = df[timepoint]/total_count


    #Calculate log2FC between the bins 
    #Get fold change between bins of interest without normalization
    df[comparison] = df[timepoint1+"_freq"]/df[timepoint2+"_freq"]

    #Get log2_FC without normalization
    df["log2FC_"+comparison] = np.log2(df[comparison])
    df.set_index('OligoID').to_csv(run_prefix+"/"+comparison + "/" + run_prefix+"_D0_skew_comparison.txt",sep='\t')





    df = pd.read_csv(run_prefix+"/"+comparison+"/"+run_prefix+"_D0_skew_comparison.txt",sep='\t')
    non_targeting_name = "non-targeting"
    safe_targeting_name = "safe-targeting"
    p_values = []
    target_list = []
    log2FC_list = []
    
    non_targeting_df = df[df['Target']==non_targeting_name]
    safe_targeting_df = df[df['Target']==safe_targeting_name]
    ctrl_dataframe = pd.concat([non_targeting_df, safe_targeting_df], ignore_index=True)
    controls = ctrl_dataframe["log2FC_D0_vs_Skew"]
    

    # Iterate over the targets and perform a two-sample t-test
    for target in df['Target'].unique():    
        targeting = df.loc[df['Target'] == target]["log2FC_D0_vs_Skew"]
        #####################
        u_stat, p_value = mannwhitneyu(controls, targeting)
        p_values.append(p_value)
        target_list.append(target)
        log2FC_list.append(targeting.mean())
    
    combined_target_significance = pd.DataFrame({'Target':target_list,'p_value':p_values,'log2FC':log2FC_list})
        
    combined_target_significance.set_index('Target').to_csv(run_prefix+ "/" +comparison+"/"+run_prefix+"_"+comparison+"_unadj_t_test.txt",sep='\t')

    

    bh_p_val_adj(run_prefix,timepoint1,timepoint2)

    show_volcano_plot(run_prefix,timepoint1,timepoint2,not_include_targets,p_thresh)
        
    list_targets = get_hits(timepoint1,timepoint2,run_prefix)
    df = pd.DataFrame(list_targets, columns=['Hits'])
    df.to_csv(run_prefix + '/' + timepoint1 + '_vs_' + timepoint2 + '_list_of_hits.txt',sep='\t')

    generate_target_barplot_summaries_target_specific_skew(timepoint1,timepoint2,run_prefix,list_targets)
    



In [ ]:
#Notes




"""""""""
Still a lot of false positives after the chromosome filter 
Requiring a control guide to control for covariates 
Spiked in validated guides with our library 
Check hemocytomoer iamges
Check our dilutions
Check with Jesse about running D0 comparison at Broad 

"""""""""

In [ ]:
#Skew counts have to have column heads of OligoID and counts

input_sheet = pd.read_csv('Input_Sheet.txt',sep='\t')
parent_directory_path = '/Users/cjmunger/Documents/230523_Pipeline'

timepoints = ['D0','Pos','Neg']


#Generating output folders for all samples we are running

for index,row in input_sheet.iterrows():
    # Define the name of the new folder
    sample = row['Sample']
    
    # Construct the complete path for the new folder
    new_folder_path = os.path.join(parent_directory_path, sample)

    # Create the new folder
    create_folder_if_not_exists(new_folder_path)
            
    t_test_method(sample,row['PCRrepFilePath'],row['skewFilePath'],row['GuideLib'],timepoints,100)



In [ ]:
#Skew counts have to have column heads of OligoID and counts

input_sheet = pd.read_csv('Input_Sheet.txt',sep='\t')
parent_directory_path = '/Users/cjmunger/Documents/230523_Pipeline'

timepoints = ['D0','Pos','Neg']


#Generating output folders for all samples we are running

for index,row in input_sheet.iterrows():
    # Define the name of the new folder
    sample = row['Sample']
    
    # Construct the complete path for the new folder
    new_folder_path = os.path.join(parent_directory_path, sample)

    # Create the new folder
    create_folder_if_not_exists(new_folder_path)
            
    u_test_method(sample,row['PCRrepFilePath'],row['skewFilePath'],row['GuideLib'],timepoints,100)



In [ ]:
####Testing pathway enrichment
run_prefix = 'combined'
timepoint1 = 'Pos'
timepoint2 = 'Neg'



#kegg_db = KEGGDatabase()

gene_list = pd.read_csv(run_prefix + '/' + timepoint1 + '_vs_' + timepoint2 + '_list_of_hits.txt',sep='\t')['Hits'].tolist()
enriched_pathways = enrich(gene_list, 'KEGG_2019_Human')

for pathway in enriched_pathways:
    print(f"Pathway: {pathway.pathway_id}")
    print(f"Description: {pathway.description}")
    print(f"p-value: {pathway.p_value}")
    print(f"q-value (FDR-adjusted p-value): {pathway.q_value}")
    print("---")
    
    
    


In [ ]:
#Updating target file 
temp_df = pd.read_csv('Data/guideLibs/genome_wide_guide_lib.txt',sep='\t')
df = temp_df[['OligoID']]
import re

df['Target'] = df['OligoID'].apply(lambda x: x.split('-', 1)[0].split('+', 1)[0] + '-' + x.rsplit('-', 1)[-1])

df.to_csv('genome_wide_guide_lib.txt',sep='\t')

#Generating skew files 

def normalize_df(df, column):
    total = df[column].sum()
    df[column] = df[column].apply(lambda x: x/total)
    return df


def multiply_and_round_column(df, column,multiplier):
    df[column] = (df[column] * multiplier).astype(int)
    return df


#Normalized (combining assuming the coverage from bacterial dilutions is accurate)
output_name = 'combined_coverage_norm'

#Merging with total guides to find those missing
temp_1 = pd.read_csv('Data/skewResults/Genome_Wide_1_23_23.count.txt',sep='\t',header=None)
temp_2 = pd.read_csv('Data/skewResults/Genome_Wide_25_Lib.count.txt',sep='\t',header=None)
temp_3 = pd.read_csv('Data/skewResults/Genome_Wide_50_Lib.count.txt',sep='\t',header=None)

temp_1.columns = ['count','OligoID']
temp_2.columns = ['count','OligoID']
temp_3.columns = ['count','OligoID']

temp_df = pd.read_csv('Data/guideLibs/genome_wide_guide_lib.txt',sep='\t')
temp_df = temp_df[['OligoID']]

temp_1 = pd.merge(temp_df,temp_1,on='OligoID',how='outer').fillna(0)
temp_2 = pd.merge(temp_df,temp_2,on='OligoID',how='outer').fillna(0)
temp_3 = pd.merge(temp_df,temp_3,on='OligoID',how='outer').fillna(0)



multiplier_1 = 5000000
multiplier_2 = 1000000
multiplier_3 = 1700000

temp_df = pd.merge(temp_1,temp_2,on='OligoID',how='outer').fillna(0)
combined_df = pd.merge(temp_df,temp_3,on='OligoID',how='outer').fillna(0)


for col in ['count_x','count_y','count']:
    combined_df = normalize_df(combined_df,col)
    
combined_df['total_count'] = multiplier_1 * combined_df['count_x'] + multiplier_2 * combined_df['count_y'] + multiplier_3 * combined_df['count']

combined_df = multiply_and_round_column(combined_df[['total_count','OligoID']],'total_count',10)


combined_df.set_index(['total_count'])[['OligoID']].to_csv(output_name + '.count.txt',sep='\t',header=None)




In [ ]:
#Generate combined PCR rep results 


list_subsets = ['first','second']

pcrrep_folder = 'Data/firstSeq'
reps = 141


pcrrep_folder = 'Data/' + 'first' + 'Seq/byPCRRep/'
pcr_rep_1 = pd.read_csv(pcrrep_folder+'MACS-Rep1-1-PCR1.bin_counts.txt',sep='\t')
total_df = pcr_rep_1[['OligoID','Pos','Neg','D0']]

for rep in range(reps):
    print(rep)
    pcrrep_folder = 'Data/' + 'first' + 'Seq/byPCRRep/'
    pcr_rep_1 = pd.read_csv(pcrrep_folder+'MACS-Rep1-1-PCR1.bin_counts.txt',sep='\t')
    pcrrep_folder = 'Data/' + 'second' + 'Seq/byPCRRep/'
    pcr_rep_2 = pd.read_csv(pcrrep_folder+'MACS-Rep1-1-PCR1.bin_counts.txt',sep='\t')

    # Merge dataframes on 'OligoID' column
    merged_df = pd.merge(pcr_rep_1, pcr_rep_2, on='OligoID', suffixes=('_df1', '_df2'),how='outer')
    merged_df.fillna(0)
    
    # Perform column-wise summation
    merged_df['D0_sum'] = merged_df['D0_df1'] + merged_df['D0_df2']
    merged_df['Pos_sum'] = merged_df['Pos_df1'] + merged_df['Pos_df2']
    merged_df['Neg_sum'] = merged_df['Neg_df1'] + merged_df['Neg_df2']
    
    total_df['Pos'] = total_df['Pos'] + merged_df['']


In [ ]:
timepoint1 = 'Pos'
timepoint2 = 'Neg'
prefix = 'first'
list_targets = ['CDH5_-P1P2']



timepoint1 = 'Pos'
timepoint2 = 'Neg'
comparison = timepoint1 + "_vs_" + timepoint2
pdf_title = prefix+'/'+comparison+"/"+prefix+"_" + timepoint1 + "_vs_" + timepoint2 + "_target_freq_swarms.pdf"
pdf = matplotlib.backends.backend_pdf.PdfPages(pdf_title)  

if timepoint2 != 'Skew':
    df = pd.read_csv(prefix+'/'+comparison+"/"+prefix+"_" + timepoint1 + "_vs_" + timepoint2 + "_normalized_dataframe.txt",sep='\t')
else:
    df = pd.read_csv(prefix+'/'+comparison+"/"+prefix+"_" + timepoint1 + "_vs_" + timepoint2 + "_normalized_dataframe.txt",sep='\t')
common_suffix = '_D0_ctrl_norm'

for target in list_targets:
    print(target)
    a4_dims = (20, 8.27)
    fig, ax = plt.subplots(figsize=a4_dims)

    # Define the patterns to match
    patterns = [target, 'non-targeting','safe-targeting']

    # Create a regular expression pattern using the patterns list
    pattern = '|'.join(patterns)

    # Filter the DataFrame based on the 'Target' column
    target_df = df[df['Target'].str.contains(pattern)][['OligoID','Target','D0'+common_suffix,'Pos'+common_suffix,'Neg'+common_suffix,'EC_total'+common_suffix]]
    #target_df = df[df['Target'].str.contains(target)][['OligoID','Target','D0'+common_suffix,'Pos'+common_suffix,'Neg'+common_suffix,'EC_total'+common_suffix]]

    if len(target_df)>0:

        column_mapping = {'OligoID': 'OligoID', 'Target': 'Target', 'D0'+common_suffix: 'D0', 'Pos'+common_suffix: 'Pos', 'Neg'+common_suffix: 'Neg','EC_total'+common_suffix: 'EC_total',}
        target_df = target_df.rename(columns=column_mapping)

        target_df = target_df.melt(id_vars=['OligoID', 'Target'], var_name='Timepoint', value_name='Frequency')
        ax = sns.barplot(data=target_df,x="Target", y="Frequency", hue="Timepoint",alpha=0.2,ci='sd',capsize=0.1)
        sns.swarmplot(x="Target", y="Frequency", data=target_df, hue="Timepoint",dodge=True,ax=ax)
        ax.set_title(target,fontsize=20)
        ax.set_ylabel('Normalized Frequency')
        plt.show()
    else:
        print('Missing ' + target)



In [ ]:
###Generating combined PCR reps
for rep in range(141):
    rep=rep+1
    df1 = pd.read_csv('Data/firstSeq/byPCRRep/MACS-Rep1-1-PCR' + str(rep) + ".bin_counts.txt",sep='\t')
    df2 = pd.read_csv('Data/secondSeq/byPCRRep/MACS-Rep1-1-PCR' + str(rep) + ".bin_counts.txt",sep='\t')

    merged_df = pd.merge(df1, df2, on='OligoID', how='outer')
    merged_df.fillna(0,inplace=True)
    for timepoint in timepoints:
        x_label = timepoint + "_x"
        y_label = timepoint + "_y"
        if x_label in merged_df.columns and y_label in merged_df.columns:
            merged_df[timepoint] = merged_df[x_label] + merged_df[y_label]
    for timepoint in timepoints:
        if timepoint not in merged_df.columns:
            merged_df[timepoint] = 0
    merged_df = merged_df[['OligoID','D0','Pos','Neg']]
    merged_df.set_index('OligoID').to_csv('Data/combinedSeq/byPCRRep/MACS-Rep1-1-PCR' + str(rep) + '.bin_counts.txt',sep='\t')

#Separating into separate files for counts

for rep in range(141):
    rep=rep+1
    df = pd.read_csv('Data/combinedSeq/byPCRRep/MACS-Rep1-1-PCR' + str(rep) + ".bin_counts.txt",sep='\t')
    for timepoint in timepoints:
        df[['OligoID',timepoint]].set_index('OligoID').to_csv('Data/combinedSeq/counts/'+ timepoint + "_" + str(rep) + '.count.txt',sep='\t',header=False)





In [ ]:
def prepare_mageck_input(df,prefix):
    file_name = prefix+'_mageck_input.txt'
    df = df.rename(columns={'OligoID': 'sgRNA', 'Target': 'gene'})
    df.set_index('sgRNA').to_csv(file_name,sep='\t')
    
    timepoints = df.columns.tolist()
    timepoints.remove('sgRNA')
    timepoints.remove('gene')
    
    
    comparisons = []
    
    for i in range(len(timepoints)):
        for j in range(i+1, len(timepoints)):
            comparison = (timepoints[i], timepoints[j])
            comparisons.append(comparison)
    
    
    commands = []
    for comparison in comparisons:
        commands.append('mageck test -k ' + file_name + ' -t ' + comparison[0] + ' -c ' + comparison[1] + " --control-sgrna " + 'non_targeting_sgRNAs.txt' ' -n ' + prefix + "_" + comparison[0] + "_" + comparison[1])
    
    # Define the name of the .sh file to be created
    filename = prefix+'_mageck_commands.sh'

    # Write the commands to the .sh file
    with open(filename, 'w') as f:
        for command in commands:
            f.write(command + '\n')

    # Make the .sh file executable
    os.chmod(filename, 0o755)

    # Execute the .sh file
    #os.system('./{}'.format(filename))

    return df



def run_mageck_on_bins(prefix):
    df = pd.read_csv(prefix + '/' + prefix + '_total_counts.txt',sep='\t')
    
    
    
    file_name = prefix+'_mageck_input.txt'
    df = df.rename(columns={'OligoID': 'sgRNA', 'Target': 'gene'})
    df.set_index('sgRNA').to_csv(file_name,sep='\t')
    
    timepoints = df.columns.tolist()
    timepoints.remove('sgRNA')
    timepoints.remove('gene')
    
    
    comparisons = []
    
    for i in range(len(timepoints)):
        for j in range(i+1, len(timepoints)):
            comparison = (timepoints[i], timepoints[j])
            comparisons.append(comparison)
    
    
    commands = []
    for comparison in comparisons:
        folder_path = prefix + '/mageck/' + comparison[0] + '_vs_' + comparison[1] + "/" + comparison[0] + '_vs_' + comparison[1]
        create_folder_if_not_exists(folder_path)
        commands.append('mageck test -k ' + file_name + ' -t ' + comparison[0] + ' -c ' + comparison[1] + " --control-sgrna " + 'Data/guideLibs/non_targeting_sgRNAs.txt' ' -n ' + folder_path)
    
    # Define the name of the .sh file to be created
    filename = prefix+'_mageck_commands.sh'

    # Write the commands to the .sh file
    with open(filename, 'w') as f:
        for command in commands:
            f.write(command + '\n')

    # Make the .sh file executable
    os.chmod(filename, 0o755)

    # Execute the .sh file
    #os.system('./{}'.format(filename))

    return df

prefix = 'combined_utest_P1P2'
df = pd.read_csv(prefix + "/" + prefix + "_total_counts.txt",sep='\t')
prepare_mageck_input(df,prefix)

run_mageck_on_bins(prefix)




In [ ]:
#####Updating guide target file 
import re

temp = pd.read_csv('Data/guideLibs/genome_wide_guide_lib.txt',sep='\t')



# Define the function to extract the text
def extract_text(input_string):
    pattern = r"(.*?)_(?:\-|\+)\_"
    match = re.search(pattern, input_string)
    if match:
        return match.group(1)
    else:
        return input_string
    
    
temp['Target'] = temp['OligoID'].apply(lambda x: extract_text(x))

temp.loc[temp['OligoID'].str.contains('non-targeting'), 'Target'] = 'non-targeting'
temp.loc[temp['OligoID'].str.contains('safe-targeting'), 'Target'] = 'safe-targeting'


temp.set_index('chr').to_csv('test.txt',sep='\t')
